# Qwen3-TTS Cross-Platform User-Story Validation

Validates the **torch + CUDA** half of the platform matrix — the one place
it can be proven, since neither macOS/MLX nor a Linux CPU container has a
GPU.

This notebook is deliberately **separate** from `colab_notebook.ipynb`: that
one is a user-facing demo that mounts Drive, this one clones from GitHub and
is a test harness.

**Before running:** set `BRANCH` in the clone cell to the branch under test.
Opening this file from a branch in the Colab GitHub picker does **not**
change what `git clone` fetches — without setting `BRANCH` you will validate
`main` and believe you validated your branch.

**Runtime:** Runtime → Change runtime type → T4 GPU (or L4 on Pro), then
Runtime → Run all. Paste back the output of any cell that errors.

## 1. Clone Repository and Install Dependencies

In [ ]:
# Set this to the branch under test. The Colab GitHub picker only chooses
# which FILE to open — it does not affect what this clone fetches.
BRANCH = "main"  # @param {type:"string"}

REPO = "https://github.com/eepstein201/Qwen3-TTS-Advanced-EME.git"
!git clone --branch $BRANCH --single-branch $REPO
%cd Qwen3-TTS-Advanced-EME
!git log --oneline -1

In [ ]:
# Install test dependencies
!pip install -e ".[test]" --quiet

## 2. Verify Installation

In [ ]:
import sys
print(f"Python: {sys.version}")

# Check for GPU (Colab Pro/L4)
try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch not installed (CPU Colab)")

## 3. Full non-E2E suite

The superset the CI `coverage` job runs. `run_batches.py` is an explicit
allowlist and silently skips any module not registered in `BATCHES`, so it
is not sufficient on its own.

In [ ]:
# --continue-on-collection-errors so ONE environment-specific import
# failure cannot abort all ~2860 tests. Collection errors still appear in
# the summary line as 'N error', so nothing is hidden by this flag.
!python -m pytest tests/ -q -m "not e2e" --continue-on-collection-errors --tb=short

## 4. Static gates

In [ ]:
!ruff check qwen3_tts tests
!mypy qwen3_tts/core qwen3_tts/server qwen3_tts/interface
!bandit -r qwen3_tts -c pyproject.toml -ll
!python -m qwen3_tts.tools.check_config_docs

## 5. Config reconciliation

Guards the class of bug where a hand-maintained config template silently
reverts a changed default.

In [ ]:
# install.sh writes config.json from a heredoc rather than calling
# get_default_config(), so the two can drift without any test noticing.
# That is exactly how the language default got reverted to English on
# every fresh install.
from qwen3_tts.core.config import get_default_config, load_config

defaults = get_default_config()
assert defaults["language"] == "auto", defaults["language"]
assert defaults["default_clone_prompt"] is None
assert load_config().get("language") == "auto", "config.json drifted from the code default"

with open("install.sh") as f:
    install_text = f.read()
assert '"language": "auto"' in install_text, "install.sh template drifted"
assert '"language": "English"' not in install_text
assert '"default_clone_prompt": "default_clone.pt"' not in install_text
print("config defaults reconciled")

## 6. Reference-audio sample-rate guarantee

The MLX runaway bug cannot reproduce here (torch backend), but the
**write-side guarantee** must hold on every platform.

In [ ]:
import numpy as np

from qwen3_tts.core.engine.audio_processing import (
    DEFAULT_SAMPLE_RATE,
    ensure_min_sample_rate,
)

t = np.linspace(0, 2.0, 16000, endpoint=False)
low = (0.3 * np.sin(2 * np.pi * 220 * t)).astype(np.float32)

out, sr, resampled = ensure_min_sample_rate(low, 8000)
assert resampled and sr == DEFAULT_SAMPLE_RATE, (sr, resampled)
assert abs(len(out) / sr - 2.0) < 0.05, "duration not preserved"

# 48 kHz must pass through untouched — never downsample.
high = (0.3 * np.sin(2 * np.pi * 220 * np.linspace(0, 1, 48000))).astype(
    np.float32
)
_, hsr, hres = ensure_min_sample_rate(high, 48000)
assert hsr == 48000 and not hres

# Stereo must be reduced to mono even when the rate needs no change.
stereo = np.stack([high, high], axis=-1)
mono_out, msr, _ = ensure_min_sample_rate(stereo, 48000)
assert mono_out.ndim == 1, "stereo reference passed through"

print(f"sample-rate guarantee holds: 8000 -> {sr} Hz, 48000 preserved, stereo reduced")

## 7. Real generation on CUDA

Hollow-green guard: assert the **artifact** (real, non-silent audio at the
expected rate), not an HTTP 200.

In [ ]:
import subprocess
import tempfile
import time

import numpy as np
import soundfile as sf

from qwen3_tts.server.client import TTSClient

server = subprocess.Popen(["tts", "server", "start", "--foreground"])
client = TTSClient()
for _ in range(120):
    if client.is_server_running():
        break
    time.sleep(5)
else:
    raise RuntimeError("server never came up — check the log above")

outdir = tempfile.mkdtemp()
for mode, kwargs in [
    ("design", {"description": "A calm, friendly voice"}),
    ("custom", {"speaker": "ryan"}),
]:
    client.load_model(mode)
    path = f"{outdir}/{mode}.wav"
    # generate() writes the file and returns its path — it does NOT
    # return a dict. Assert on the audio actually written to disk.
    client.generate(
        "Cross platform verification passed.",
        output=path,
        mode=mode,
        **kwargs,
    )
    wav, sr = sf.read(path)
    peak = float(np.abs(wav).max())
    dur = len(wav) / sr
    assert sr == DEFAULT_SAMPLE_RATE, sr
    assert dur > 0.5, f"{mode}: suspiciously short ({dur:.2f}s)"
    assert peak > 0.01, f"{mode}: silent output"
    print(
        f"{mode}: {dur:.1f}s @ {sr} Hz, peak {peak:.3f}, seed {client.last_seed}"
    )

## 8. CLI user-story sweep

Server-independent surface. `tts <text>` runs models **in-process** by
default (`use_server` is only true with the hidden `--_server-mode` flag),
so these exercise the local path.

In [ ]:
for cmd in [
    "tts --help", "tts list speakers", "tts list presets", "tts list aliases",
    "tts list prosody", "tts list backends", "tts list models",
    "tts config show", "tts config path", "tts voice list",
    "tts cache size", "tts doctor",
]:
    print(f"\n$ {cmd}")
    !{cmd}

## 9. What this run proves

| Section | Proves |
|---|---|
| 3 | Full non-E2E suite on torch/CUDA — the superset CI's coverage job runs, not the `BATCHES` allowlist subset |
| 4 | ruff / mypy / bandit / config-docs gates hold on Linux+CUDA |
| 5 | `install.sh`, `config.json` and `get_default_config()` agree — no silent template drift |
| 6 | Reference audio below 24 kHz is upsampled on write, 48 kHz is preserved, stereo is reduced to mono |
| 7 | Real non-silent audio at the expected rate in design and custom modes, via a live server on a real GPU |
| 8 | The read-only CLI surface runs on this platform |

A failure in any section is a real finding — paste the cell output back.

**Not proven here:** macOS/MLX behavior (including the MLX runaway-generation
bug itself, which cannot reproduce on the torch backend) and Windows.